# SmartStudy Agent: An Agentic AI Study Planner
### CSE476 Agentic AI and Intelligent Automation — CA1 Project 1
**Project Name:** SmartStudy Agent  
**Subtitle:** *An Agentic AI Study Planner*  
**Topic:** T24 — Study Planner Agent  
**Framework:** Python 3.10+ | OpenAI Function Calling API / Groq API  

---

## 1. Problem Statement & Motivation
University students struggle to balance multiple coursework deadlines, exams, and projects. Standard chatbots offer generic advice but cannot track deadline dates, prioritize urgent tasks deterministically, or re-plan study blocks when new deadlines are introduced.

**SmartStudy Agent** solves this by acting as a **Real AI Agent** featuring a complete **Plan-Act-Observe loop**. It combines deterministic Python tools with memory persistence to organize tasks, calculate urgency, and generate optimized study schedules.

## 2. University CA1 Requirement Checklist
- ✅ **1. One AI Agent with Plan-Act Loop:** Multi-step decision loop implemented in `agent.py`.
- ✅ **2. Takes More Than 1 Step:** Agent executes multiple tool calls sequentially per turn.
- ✅ **3. Tool-Based Decisions:** Agent chooses next action dynamically based on tool results.
- ✅ **4. At Least 2 Real Tools:** Implements `add_task(name, due)` and `build_schedule()`.
- ✅ **5. Real Tool Execution:** Tools execute concrete Python code modifying application state.
- ✅ **6. Session Memory:** `StudyAgentMemory` persists tasks, deadlines, and preferences.
- ✅ **7. Multi-Turn Persistence:** Remembers information from earlier turns in the conversation.
- ✅ **8. Memory Reading:** Reads memory in later turns to answer follow-up queries without repeating.
- ✅ **9. Interactive User Input:** Supports live interactive student prompts via CLI and notebook `input()`.
- ✅ **10. Jupyter Notebook Demos:** 3 complete demonstration scenarios.
- ✅ **11. Trace & Observability:** Clear formatted trace showing Agent Decisions, Tool Calls, Tool Results, Memory Snapshots, and Final Response.

## 3. Agent Architecture (Plan-Act-Observe Loop)

```
          +-------------------------------------------------+
          |                    USER GOAL                    |
          +-------------------------------------------------+
                                   |
                                   v
          +-------------------------------------------------+
          |              SMARTSTUDY AGENT LOOP              |
          |  (Reads System Prompt + History + Memory State) |
          +-------------------------------------------------+
                                   |
                                   v
          +-------------------------------------------------+
          |              1. AGENT DECISION                  |
          |  (Selects Tool to call or gives Final Answer)   |
          +-------------------------------------------------+
                     /                           \
       [Tool Call]  /                             \  [Final Answer]
                   v                               v
    +------------------------------+     +-------------------+
    |      2. TOOL CALL            |     |   FINAL RESPONSE  |
    | (add_task / build_schedule)  |     +-------------------+
    +------------------------------+
                   |
                   v
    +------------------------------+
    |      3. TOOL RESULT          |
    | (Executes Real Python Code)  |
    +------------------------------+
                   |
                   v
    +------------------------------+
    |   4. UPDATE SESSION MEMORY   |
    |  (tasks, deadlines, urgency) |
    +------------------------------+
                   |
                   +---> (Loop Back to AGENT DECISION for Step 2)
```

## 4. Setup & Imports

In [1]:
import sys
from pathlib import Path

# Ensure project directory is in python path
sys.path.insert(0, str(Path.cwd()))

from config import TODAY_DATE, IS_OFFLINE_MODE, MODEL_NAME
from memory import StudyAgentMemory
from tools import add_task, build_schedule
from agent import SmartStudyAgent, print_formatted_trace

print('System Loaded Successfully!')
print('Today Reference Date:', TODAY_DATE)
print('Selected LLM Model:', MODEL_NAME)
print('Offline Fallback Mode:', IS_OFFLINE_MODE)

System Loaded Successfully!
Today Reference Date: 2026-08-31
Selected LLM Model: openai/gpt-oss-120b
Offline Fallback Mode: False


## 5. Direct Tool Verification
Let's test our two tools directly to verify that they modify application state in memory.

In [2]:
# Initialize temporary memory instance
test_memory = StudyAgentMemory()

# Call Tool 1: add_task()
res1 = add_task('Python Assignment', '2026-08-26', test_memory)
print('Tool 1 (add_task) Output:')
print(res1)

# Call Tool 2: build_schedule()
res2 = build_schedule(test_memory)
print('\nTool 2 (build_schedule) Output:')
print(res2)

Tool 1 (add_task) Output:
{'success': False, 'message': "Error: Due date '2026-08-26' has already passed.", 'task': None}

Tool 2 (build_schedule) Output:
{'success': False, 'message': 'Cannot build schedule: There are currently no tasks stored in memory. Please add tasks first using add_task().', 'schedule': None}


## 6. DEMO 1 — Multi-Step Goal (Multiple Tool Calls in 1 Turn)
**User Goal:** *"I have a Python assignment due tomorrow and a DBMS exam in five days. Make a study plan."*

**Multi-Step Agent Execution Flow:**
1. `add_task("Python Assignment", "2026-08-26")` -> Memory updated.
2. `add_task("DBMS Exam", "in 5 days")` -> Memory updated.
3. `build_schedule()` -> Schedule generated.
4. Final structured study plan presented.

In [3]:
agent1 = SmartStudyAgent()
prompt1 = 'I have a Python assignment due tomorrow and a DBMS exam in five days. Make a study plan.'

response1, trace1 = agent1.run(prompt1)
print_formatted_trace(prompt1, response1, trace1)

[Agent Error] Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}}. Falling back to offline planner.
                      AGENT TRACE                       

[USER INPUT]: I have a Python assignment due tomorrow and a DBMS exam in five days. Make a study plan.

------------------------------------------------------------

[STEP 1: AGENT DECISION]
  Decision: Offline Planner decided to run 'add_task'.
  Tool Executed: add_task
  Arguments Passed: {"name": "Python Assignment", "due": "tomorrow"}

[STEP 1: TOOL RESULT]
  Tool Name: add_task
  Output: {
  "success": true,
  "message": "Task 'Python Assignment' successfully added to memory (due 2026-09-01, 1 days left).",
  "task": {
    "name": "Python Assignment",
    "due": "2026-09-01",
    "days_left": 1
  }
}
  [MEMORY SNAPSHOT AFTER TOOL]:
    Total Tasks: 1
      1. Python Assignment (Due: 2026-09-01, Days Left: 1)

[STEP 2: AGENT DECISION]
  Decision: Offline Plann

## 7. DEMO 2 — Session Memory Across Multi-Turn Conversation
**Goal:** Prove memory persistence across turns.
- **Turn 1:** *"I have an AI assignment due tomorrow."* -> Agent stores task.
- **Turn 2:** *"What should I prioritize today?"* -> Agent retrieves task from memory and answers without requiring the user to repeat the task name!

In [1]:
agent2 = SmartStudyAgent()

print('=== TURN 1 ===')
prompt2_t1 = 'I havean AI assignment due tomorrow.'
resp2_t1, trace2_t1 = agent2.run(prompt2_t1)
print_formatted_trace(prompt2_t1, resp2_t1, trace2_t1)

print('\n=== TURN 2 (Follow-up) ===') 
prompt2_t2 = 'What should I prioritize today?'
resp2_t2, trace2_t2 = agent2.run(prompt2_t2)
print_formatted_trace(prompt2_t2, resp2_t2, trace2_t2)

=== TURN 1 ===
[Agent Error] Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01m0vrmz6aevx8nry9ex2nwxmm` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 6968, Requested 1449. Please try again in 3.1275s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}. Falling back to offline planner.
                      AGENT TRACE                       

[USER INPUT]: I have an AI assignment due tomorrow.

------------------------------------------------------------

[STEP 1: AGENT DECISION]
  Decision: Decided to execute tool 'add_task' based on prompt/memory.
  Tool Executed: add_task
  Arguments Passed: {"due": "tomorrow", "name": "AI assignment"}

[STEP 1: TOOL RESULT]
  Tool Name: add_task
  Output: {
  "success": true,
  "message": "Task 'AI assignment' successfully added to memory (due 2026-08-26, 1 days left).",
 

## 8. DEMO 3 — Dynamic Replanning
**Goal:** Show that when new urgent tasks arrive, the agent recalculates deadline priorities and updates the schedule dynamically.

In [1]:
agent3 = SmartStudyAgent()

# Step 1: Initial schedule
agent3.run('I have a DBMS exam in 5 days. Make me a study plan.')
print('INITIAL MEMORY TASKS:')
print(agent3.memory.get_memory_summary())

# Step 2: Introduce new urgent deadline
prompt3_replan = 'I also have a Python assignment due tomorrow! Update my schedule.'
resp3_replan, trace3_replan = agent3.run(prompt3_replan)

print_formatted_trace(prompt3_replan, resp3_replan, trace3_replan)

INITIAL MEMORY TASKS:
Total Tasks: 1
  1. DBMS Exam (Due: 2026-08-30, Days Left: 5)
                      AGENT TRACE                       

[USER INPUT]: I also have a Python assignment due tomorrow! Update my schedule.

------------------------------------------------------------

[STEP 1: AGENT DECISION]
  Decision: Decided to execute tool 'add_task' based on prompt/memory.
  Tool Executed: add_task
  Arguments Passed: {"due": "tomorrow", "name": "Python Assignment"}

[STEP 1: TOOL RESULT]
  Tool Name: add_task
  Output: {
  "success": true,
  "message": "Task 'Python Assignment' successfully added to memory (due 2026-08-26, 1 days left).",
  "task": {
    "name": "Python Assignment",
    "due": "2026-08-26",
    "days_left": 1
  }
}
  [MEMORY SNAPSHOT AFTER TOOL]:
    Total Tasks: 2
      1. DBMS Exam (Due: 2026-08-30, Days Left: 5)
      2. Python Assignment (Due: 2026-08-26, Days Left: 1)

[STEP 2: AGENT DECISION]
  Decision: Decided to execute tool 'build_schedule' based on pro

## 9. Failure & Edge Case Handling (Honest Edge Case)
**Scenario:** User requests a schedule when **no tasks exist** in memory.
**Expected Behavior:** `build_schedule()` returns structured failure result (`success: False`), informing the user to add tasks first.

In [1]:
empty_agent = SmartStudyAgent()
prompt_empty = 'Make my study schedule.'

resp_empty, trace_empty = empty_agent.run(prompt_empty)
print_formatted_trace(prompt_empty, resp_empty, trace_empty)

                      AGENT TRACE                       

[USER INPUT]: Make my study schedule.

------------------------------------------------------------

[STEP 1: FINAL RESPONSE]
I’d love to put together a focused study schedule for you! To make it as useful as possible, could you let me know the upcoming assignments, exams, projects, or any other academic tasks you need to prepare for (including their due dates or exam dates)? Once I have those details, I’ll add them to your task list and generate a prioritized study plan. 🌟




## 10. Interactive User Input Demo
You can test custom student prompts interactively below!

In [1]:
# Interactive notebook input
interactive_agent = SmartStudyAgent()
sample_input = "I have a Machine Learning Lab due in 3 days and a Web Dev Project due in 7 days. Create my study schedule."
print("Running custom prompt:", sample_input)
interactive_resp, interactive_trace = interactive_agent.run(sample_input)
print_formatted_trace(sample_input, interactive_resp, interactive_trace)

Running custom prompt: I have a Machine Learning Lab due in 3 days and a Web Dev Project due in 7 days. Create my study schedule.
                      AGENT TRACE                       

[USER INPUT]: I have a Machine Learning Lab due in 3 days and a Web Dev Project due in 7 days. Create my study schedule.

------------------------------------------------------------

[STEP 1: AGENT DECISION]
  Decision: Decided to execute tool 'add_task' based on prompt/memory.
  Tool Executed: add_task
  Arguments Passed: {"due": "in 3 days", "name": "Machine Learning Lab"}

[STEP 1: TOOL RESULT]
  Tool Name: add_task
  Output: {
  "success": true,
  "message": "Task 'Machine Learning Lab' successfully added to memory (due 2026-08-28, 3 days left).",
  "task": {
    "name": "Machine Learning Lab",
    "due": "2026-08-28",
    "days_left": 3
  }
}
  [MEMORY SNAPSHOT AFTER TOOL]:
    Total Tasks: 1
      1. Machine Learning Lab (Due: 2026-08-28, Days Left: 3)

[STEP 2: AGENT DECISION]
  Decision: Deci

## 11. Conclusion & Viva Summary
SmartStudy Agent demonstrates a complete, viva-ready Agentic AI solution with real tools, session memory, multi-step decision loops, and complete trace observability.